<center>
    <h2>Evaluation protocole for the 2026 IA Pau Data Battle</h2>
</center>

---

This notebook provides you a protocol to evaluate your model for the data battle


### Why this notebook

This is not:

- The only and perfect way to evaluate your model.

The subject of evaluation is not trivial and there are different methods with their pro and con. The construction of the relevant method for evaluation is <span style="color:red">part of the subject of the data battle</span>. For instance, proposing a new relevant version of the risk estimation compatible with AI models would be very interesting.

- A mandatory step to participate to the Jury. You do not have to use this notebook to evaluate your models (but you may want). 

This is for :

- To provide a common evaluation framework. This enables the jury to better understand your results on the metric they understand and will see in other groups.

- Provide trust : we will send you blind test data which consists of subset of alerts. We will use the procedure described in this notebook to evaluate your results based on the prediction you will send to us. This enables us to validate your prediction.
 

### Two criteria : Risk and time gain 

As you know, our data from Meteorage consists of the location and date of lightning strikes from thunderstorms passing around areas to be protected (airports).

We assume :

- The last lightning of the alert $a$ is at time $t^a$ (all times and dates are in minutes)

- That a model produces $N^a$ predictions $p^a_i, i=1..N^a$ for the end of alert $a$. Each prediction $p^a_i$ is composed of a confidence score $s^a_i$, the date at which the prediction is emitted $d_i^a$ and the value of the prediction itself $\hat{t}_i^a$ which is the predicted time of the last lightning in the alert.

Now, we will estimate two criteria : Risk and time gain.

**A gain of time** $g_i^a$ for the prediction $p^a_i$ is measured as compared with the 30 minutes baseline :

$$g_i^a = t^a + 30 - \hat{t}_i^a$$

In practice, the overall Gain $G$ of a model will be the sum of gain over all alert contained in the dataset.


**Another important aspect is the risk** (_which measures how bad it is to be wrong_) of a lightning occurring on the airport after the end of the alert. There are multiple definitions for a relevant risk. We chose to consider the number of lightnings occurring within 3 km around the airport after the end of the alert. In practice, the risk associated to a model is computed on some training data.

$$R = \frac{M^{L3}}{N^{L3}}$$

Where the lightnings of the training data are denoted as $L = \{l_i, i=1..N^L\}$ and each lightning $l_i=(d_i,t_i)$. $N^{L3}$ is the number of lightning within a 3 kilometers distance and is computed as :

$$N^{L3} = \sum_i \mathbb{1}_{d_i<3}$$

$M^{L3} $ denotes the number of missed lightning (that is to say occurring after the end of the alarm) within a 3 kilometers distance. It is computed as

$$M^{L3} = \sum_i \mathbb{1}_{d_i<3\text{ } \& \text{ }\hat{t}_i^a < t_i \text{ }\& \text{ alert}(t_i) == a } $$

with alert$(t_i)$ returns the index of the alert for the lightning $t_i$.

Intuitively, a model $m$ that satisfies $R < 0.05$ ensures that less than 5% of lightning strikes will occur outside of alert periods. 

### How are combined the two criteria, ie, risk and time gain ($R$ and $G$)

The evaluator will set an acceptable risk value : $R_{accept}=0.02$.


To compute this risk, we must somehow select a prediction $p^a_i$ from the model. For this, we define a threshold $\theta$ and we select the predicted date $\hat{t^a}$ which verifies : 

$$\hat{t}^a = min_i\text{  }  \hat{t}_i^a, s^a_i > \theta $$

Given the selected prediction $\hat{t^a}$, we can compute a gain $g^a=t^a + 30 - \hat{t}^a$. 

To set the value of the threshold $\theta$, we test different values, and, among the values which provide a $R<R_{accept}$, we select the one with the highest gain $g^a$


### Provided python implementation

The rest of the notebook is organised as follows :

- Part 1 will generate fake predictions so that we can simulate an evaluation

- Part 2 will test different threshold $\theta$ and select the one with maximum gain $G$ while respecting the contraint on the risk $R$.

- Part 3 will evaluate the model given a supplied Theta. This last part will correspond to the code we will run with your submission.

In [1]:
import pandas as pd
## here replace with your own file
input_file = '..\\data\\segment_alerts_all_airports_eval.csv'
df = pd.read_csv(input_file)

In [2]:
init_vars = df.columns
init_vars = list(init_vars)

## Importations

### Ajout des features

In [28]:
import sys
import os
from importlib import reload

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))
from src import feature_engineering_function
reload(feature_engineering_function)
from src.feature_engineering_function import build_features

df,VAR1, TARGET1, IDS1, new_dummies1 = build_features(df)

✅ Typage & tri
✅ Variables temporelles
✅ Délais passés + futur strict
✅ Comptages rolling
✅ Comptages par type
✅ Taux d'activité
✅ Variables spatiales & azimut
✅ Variables amplitude
✅ Variables alerte
✅ Dynamique orage
✅ Silence & direction
✅ Centre de masse
✅ Cible créée | 507,071 lignes conservées


In [29]:
VAR = ['min_dist_5min',
 'time_since_last_CG20_2',
 'log_count_30min',
 'amplitude_change',
 'activity_decay',
 'time_since_last_intra_cloud2',
 'log_cg_count_10min',
 'min_dist_1min',
 'cg_20km',
 'log_std_amplitude_10min',
 'is_cloud_ground',
 'log_cg_count_20min',
 'burst_indicator',
 'hour',
 'storm_direction_change',
 'time_since_last_cloud_ground2',
 'std_lat_10min',
 'mean_dist_1min',
 'dist',
 'log_count_5min',
 'alert_duration',
 'delta_dist',
 'log_ic_count_10min',
 'storm_center_distance',
 'silence_30min',
 'azimuth_diff',
 'max_amplitude_10min',
 'std_azimuth_10min',
 'storm_spread',
 'month',
 'mean_azimuth_1min',
 'log_count_20min',
 'std_azimuth_1min',
 'azimuth',
 'storm_center_velocity',
 'log_ic_count_5min',
 'max_amplitude_1min',
 'storm_velocity',
 'mean_dist_10min',
 'storm_center_move',
 'mean_azimuth_10min',
 'mean_amplitude_1min',
 'cg_ratio',
 'rate_trend',
 'log_ic_count_20min',
 'min_dist_10min',
 'azimuth_change',
 'distance_trend',
 'mean_dist_5min',
 'mean_amplitude_10min',
 'std_lon_10min',
 'time_since_last_lightning2',
 'log_count_10min',
 'log_cg_count_5min',
 'activity_acceleration',
 'log_count_1min']

In [30]:
import joblib
artefacts = joblib.load('../models/xgb_cg10_artefacts.pkl')
model_xgb10   = artefacts['model']
xgb_vars10    = artefacts['vars_to_use']
# params      = artefacts['best_params']
imputer10     = artefacts['imputer']
# breaks      = artefacts['breaks']
# bin_stats   = artefacts['bin_stats']
# performance = artefacts['performance']
#######################################################
artefacts = joblib.load('../models/xgb_cg30_artefacts.pkl')
model_xgb30   = artefacts['model']
xgb_vars30    = artefacts['vars_to_use']
# params30      = artefacts['best_params']
imputer30     = artefacts['imputer']
#######################################################3
artefacts = joblib.load('../models/xgb_cg15_artefacts.pkl')
model_xgb15   = artefacts['model']
xgb_vars15   = artefacts['vars_to_use']
# params15     = artefacts['best_params']
imputer15     = artefacts['imputer']
#######################################################
artefacts = joblib.load('../models/xgb_cg15_3km_artefacts.pkl')
model_xgb15_3   = artefacts['model']
xgb_vars15_3   = artefacts['vars_to_use']
# params15_3     = artefacts['best_params']
imputer15_3     = artefacts['imputer']


In [33]:
df[VAR] = imputer10.fit_transform(df[VAR])
df_enc = pd.get_dummies(df[xgb_vars10])
df['y_10'] = df['time_to_next_cg20']>(10 * 60)
df['probas_10'] = model_xgb10.predict_proba(df_enc)[:, 1]

df['y_15'] = df['time_to_next_cg20']>(15 * 60)
df['probas_15'] = model_xgb15.predict_proba(df_enc)[:, 1]

df['y_30'] = df['time_to_next_cg20']>(30 * 60)
df['probas_30'] = model_xgb30.predict_proba(df_enc)[:, 1]

df_enc2 = pd.get_dummies(df[xgb_vars15_3])
df['y_15_3k'] = df['time_to_next_cg3']>(15 * 60)
df['probas_15_3km'] = 1- model_xgb15_3.predict_proba(df_enc2)[:, 1]

### Evaluation des modèles

In [34]:
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, average_precision_score,
                             brier_score_loss)
import numpy as np
def evaluate_models(df, models_config=None):
    """
    Évalue les performances de chaque modèle binaire et retourne un DataFrame synthétique.

    Parameters
    ----------
    df : pd.DataFrame
        DataFrame contenant les colonnes y_* et probas_*.
    models_config : list[dict], optional
        Liste de configs. Par défaut, utilise les 4 modèles standard.
        Chaque dict : {'name': str, 'y_col': str, 'proba_col': str, 'threshold': float}

    Returns
    -------
    pd.DataFrame avec une ligne par modèle et les métriques en colonnes.
    """
    if models_config is None:
        models_config = [
            {'name': 'XGB 10min',      'y_col': 'y_10',    'proba_col': 'probas_10',     'threshold': 0.5},
            {'name': 'XGB 15min',      'y_col': 'y_15',    'proba_col': 'probas_15',     'threshold': 0.5},
            {'name': 'XGB 30min',      'y_col': 'y_30',    'proba_col': 'probas_30',     'threshold': 0.5},
            {'name': 'XGB 15min 3km',  'y_col': 'y_15_3k', 'proba_col': 'probas_15_3km', 'threshold': 0.5},
        ]

    rows = []
    for cfg in models_config:
        name      = cfg['name']
        y_col     = cfg['y_col']
        proba_col = cfg['proba_col']
        threshold = cfg.get('threshold', 0.5)

        # Filtrer les NaN sur y ET probas
        mask = df[y_col].notna() & df[proba_col].notna()
        y_true = df.loc[mask, y_col].astype(int).values
        y_prob = df.loc[mask, proba_col].values
        y_pred = (y_prob >= threshold).astype(int)

        n = len(y_true)
        prevalence = y_true.mean()

        rows.append({
            'Modèle':       name,
            'N':            n,
            'Prévalence':   f'{prevalence:.1%}',
            'Accuracy':     accuracy_score(y_true, y_pred),
            'Precision':    precision_score(y_true, y_pred, zero_division=0),
            'Recall':       recall_score(y_true, y_pred, zero_division=0),
            'F1':           f1_score(y_true, y_pred, zero_division=0),
            'ROC AUC':      roc_auc_score(y_true, y_prob),
            'PR AUC':       average_precision_score(y_true, y_prob),
            'PR':       average_precision_score(y_true, y_prob)/prevalence,
            'Brier':        brier_score_loss(y_true, y_prob),
            'Seuil':        threshold,
        })

    results = pd.DataFrame(rows)

    # Formater les colonnes numériques
    fmt_cols = ['Accuracy', 'Precision', 'Recall', 'F1', 'ROC AUC', 'PR AUC', 'Brier']
    for col in fmt_cols:
        results[col] = results[col].map(lambda x: round(x, 4))

    return results

In [ ]:
SEUIL  = {'p15_3' : 0.591,
          'p30': 0.603,
          'p10' :0.585,
          'p15' : 0.58}

perf = evaluate_models(df, models_config=[
    {'name': 'XGB 10min',     'y_col': 'y_10',    'proba_col': 'probas_10',     'threshold': SEUIL['p10']},
    {'name': 'XGB 15min',     'y_col': 'y_15',    'proba_col': 'probas_15',     'threshold': SEUIL['p15']},
    {'name': 'XGB 30min',     'y_col': 'y_30',    'proba_col': 'probas_30',     'threshold': SEUIL['p30']},
    {'name': 'XGB 15min 3km', 'y_col': 'y_15_3k', 'proba_col': 'probas_15_3km', 'threshold': SEUIL['p15_3']},
])
print(perf.to_string(index=False))

       Modèle      N Prévalence  Accuracy  Precision  Recall     F1  ROC AUC  PR AUC       PR  Brier  Seuil
    XGB 10min 507071      17.4%    0.9585     0.8657  0.9016 0.8833   0.9861  0.9357 5.372307 0.0331    0.5
    XGB 15min 507071      14.5%    0.9603     0.8698  0.8540 0.8618   0.9849  0.9269 6.388389 0.0321    0.5
    XGB 30min 507071      10.7%    0.9745     0.8920  0.8667 0.8792   0.9879  0.9325 8.716863 0.0210    0.5
XGB 15min 3km 507071      78.1%    0.9698     0.9702  0.9919 0.9809   0.9853  0.9944 1.272709 0.0248    0.5
       Modèle      N Prévalence  Accuracy  Precision  Recall     F1  ROC AUC  PR AUC       PR  Brier  Seuil
    XGB 10min 507071      17.4%    0.9588     0.8946  0.8657 0.8799   0.9861  0.9357 5.372307 0.0331  0.585
    XGB 15min 507071      14.5%    0.9583     0.9019  0.7994 0.8476   0.9849  0.9269 6.388389 0.0321  0.580
    XGB 30min 507071      10.7%    0.9727     0.9172  0.8188 0.8652   0.9879  0.9325 8.716863 0.0210  0.603
XGB 15min 3km 507071      78

In [7]:
data = df[init_vars+['probas_10','probas_15','probas_30','probas_15_3km']].copy()

In [ ]:
df['time_to_next_cg20']df[df['time_to_next_cg20'].notna()]

KeyError: 'time_to_next_cg20'

### Utilities for prediction

In [8]:
from joblib import Parallel, delayed
from datetime import timedelta
import numpy as np

# ══════════════════════════════════════════════════════════════
# PARAMÈTRES
# ══════════════════════════════════════════════════════════════
SEUIL  = {'p15_3' : 0.591,
          'p30': 0.603,
          'p10' :0.585,
          'p15' : 0.58}
ALERT_DURATION = timedelta(minutes=10)
DT_INIT        = timedelta(minutes=0.5)   # attente initiale (premier éclair)
DT_WAIT        = timedelta(minutes=5)   # attente entre prédictions
GAP_ORAGE      = timedelta(hours=1)     # séparation entre deux orages

orig_idx   = data.index.values          # index originaux dans df global
dates      = pd.to_datetime(data['date']).values.astype('datetime64[ns]')
icloud_v   = data['icloud'].values
dist_v     = data['dist'].values
p30_v      = data['probas_30'].values
p15_v      = data['probas_15'].values
p10_v      = data['probas_10'].values
p15_3km_v  = data['probas_15_3km'].values
airport    = data['airport'].iloc[0]
n          = len(data)


# ── Constantes converties une seule fois ──────────────────────
_ALERT_NS  = np.timedelta64(int(ALERT_DURATION.total_seconds()), 's')
_WAIT_NS   = np.timedelta64(int(DT_WAIT.total_seconds()), 's')
_INIT_NS   = np.timedelta64(int(DT_INIT.total_seconds()), 's')
_30MIN_S   = 1800.0
_GAP_S     = GAP_ORAGE.total_seconds()

In [9]:
def simulate_alert_system(df_airport):

    
    df   = df_airport.sort_values('date').reset_index(drop=False)

    orig_idx  = df['index'].values
    dates     = pd.to_datetime(df['date']).values.astype('datetime64[ns]')
    p30_v     = df['probas_30'].values
    p15_v     = df['probas_15'].values
    p10_v     = df['probas_10'].values
    p15_3km_v = df['probas_15_3km'].values
    airport   = df['airport'].iloc[0]
    n         = len(df)

    # ① Pré-calcul du masque CG20 (évite les appels répétés)
    cg20_mask = (~df['icloud'].values.astype(bool)) & (df['dist'].values <= 20)

    # ② Pré-calcul du dernier index CG20 connu à chaque position → O(1) lookup
    last_cg20_idx = np.full(n, -1, dtype=np.int64)
    last = -1
    for k in range(n):
        if cg20_mask[k]:
            last = k
        last_cg20_idx[k] = last

    alerte_arr = np.zeros(n, dtype=np.int8)
    alert_log  = []

    # ③ set_alert vectorisé avec searchsorted → O(log n)
    def set_alert_fast(start_i, end_np):
        j_end = min(int(np.searchsorted(dates, end_np, side='right')), n)
        if start_i + 1 < j_end:
            alerte_arr[start_i + 1:j_end] = 1
        return j_end

    # ④ find_cg20 vectorisé avec searchsorted → O(log n) + np.any
    def has_cg20_in_window(t_start, t_end, from_i):
        j0 = max(int(np.searchsorted(dates, t_start, side='left')), from_i)
        j1 = int(np.searchsorted(dates, t_end, side='right'))
        return j0 < j1 and np.any(cg20_mask[j0:j1])
    def has_lightning_in_window(t_start, t_end, from_i):
        j0 = max(int(np.searchsorted(dates, t_start, side='left')), from_i)
        j1 = int(np.searchsorted(dates, t_end, side='right'))
        return j0 < j1
    
    def safe_last(next_out, current_i):
        """Retourne le dernier index couvert par l'alerte, ou None si invalide."""
        idx = next_out - 1
        if idx <= current_i or idx >= n:
            return None
        return idx


    # ── Épisodes orageux ──────────────────────────────────────
    gaps = np.diff(dates) / np.timedelta64(1, 's')
    ep_bounds  = np.where(gaps > _GAP_S)[0] + 1
    ep_starts  = np.concatenate([[0], ep_bounds, [n]])

    for ep in range(len(ep_starts) - 1):
        ep_start = int(ep_starts[ep])
        ep_end   = int(ep_starts[ep + 1])# exclu

        i              = ep_start
        alert_end      = None
        alert_start_ts = None
        last_in_alert  = None
        next_check     = dates[ep_start] + _INIT_NS

        while i < ep_end:
            t_now = dates[i]

            # RÈGLE ABSOLUE : CG ≤ 20km hors alerte
            if cg20_mask[i] and alert_end is None:
                t_end_np       = t_now + _ALERT_NS
                alert_start_ts = pd.Timestamp(t_now)
                next_out       = set_alert_fast(i, t_end_np)
                alert_end      = t_end_np
                last_in_alert = safe_last(next_out, i)
                alert_log.append({'airport': airport, 'alert_start': pd.Timestamp(t_now),
                                      'alert_end': pd.Timestamp(alert_end), 'type': 'CG20'})
                next_check = alert_end
                i += 1
                continue

            # FIN D'ALERTE
            if alert_end is not None and t_now > alert_end:
                # print(i)
                if last_in_alert is not None and last_in_alert > ep_start:# il y a eu au moins 1 éclair durant l alerte
                    if last_in_alert >= n:
                        prolonger = False
                    else:
                        prolonger = p10_v[last_in_alert] <= SEUIL['p10']
                    new_end = dates[last_in_alert] + _ALERT_NS
                    # print('haut')
                else: # pas d eclairs durant l alerte
                    # print('bas')
                    k = last_cg20_idx[i - 1] if i > ep_start else -1 # l index du dernier CG20
                    if k >= ep_start: 
                        t_last_CG  = (t_now - dates[k]) / np.timedelta64(1, 's')
                        # print('t_last_CG',t_last_CG)
                        prolonger = (
                            k >= ep_start and
                            t_last_CG < _30MIN_S
                        )# est ce que ca fait déjà 30mn?
                        time_to_add = min(ALERT_DURATION.total_seconds(),_30MIN_S-t_last_CG)
                        # print('time_to_add',time_to_add)
                        new_end = alert_end +  np.timedelta64(int(time_to_add), 's')
                    else:
                        prolonger = False
                # print(new_end )
                # print('prolonger',prolonger)
                if prolonger: 
                    #new_end = alert_end +  _ALERT_NS
                    start_i   = i-1
                    # print('start_i',start_i)
                    j_end  = set_alert_fast(start_i, new_end)
                    alert_log.append({'airport': airport, 'alert_start': pd.Timestamp(alert_end),
                                      'alert_end': pd.Timestamp(new_end), 'type': 'prolongation'})
                    alert_end     = new_end
                    last_in_alert = safe_last(j_end, i)
                    next_check = alert_end
                    continue
                else: # lever l alerte
                    # alert_log.append({'airport': airport, 'alert_start': pd.Timestamp(alert_start_ts),
                    #                   'alert_end': pd.Timestamp(alert_end), 'type': 'normale'})
                    alert_end = alert_start_ts = None
                    last_in_alert = None
                    next_check    = t_now
                    continue

            # ATTENTE
            if t_now < next_check:
                i += 1
                continue

            # ⑤ BOUCLE DE DÉCISION — étapes 1+2 fusionnées
            # le prochain viendra t il après 15mn?  30min?
            if p30_v[i] > SEUIL['p30'] and p15_v[i] > SEUIL['p15'] and p10_v[i] > SEUIL['p10'] and p15_3km_v[i] > SEUIL['p15_3']: # dans plus de 30 mn
                next_check = t_now + _INIT_NS #_WAIT_NS
                i += 1
                continue
            if p15_v[i] <= SEUIL['p15'] or p15_3km_v[i] <= SEUIL['p15_3'] or p10_v[i] <= SEUIL['p10']: # Dans moins de 15mn
                # Danger immédiat?
                if p15_3km_v[i] <= SEUIL['p15_3'] or p10_v[i] <= SEUIL['p10']:
                    t_type         = 'danger_3km' if p15_3km_v[i] <= SEUIL['p15_3'] else 'p10'
                    t_end_np       = t_now + _ALERT_NS
                    alert_start_ts = pd.Timestamp(t_now)
                    next_out = set_alert_fast(i, t_end_np)
                    alert_end     = t_end_np
                    last_in_alert = safe_last(next_out, i)
                    alert_log.append({'airport': airport, 'alert_start': alert_start_ts,
                                        'alert_end': pd.Timestamp(t_end_np), 'type': t_type})
                    i += 1
                    next_check = t_end_np
                    continue

            # p15 > seuil ou p10 > seuil : fenêtre 5min 
            start_i   = min(int(np.searchsorted(dates, t_now + _WAIT_NS, side='left')), ep_end)
            if not has_lightning_in_window(t_now, t_now + _WAIT_NS, i+1):
                
                t_end_np       = t_now + _WAIT_NS + _ALERT_NS# l alerte sera lancée à la fin des 5mn
                alert_start_ts = pd.Timestamp(t_now + _WAIT_NS)
                next_out = set_alert_fast(start_i, t_end_np)
                alert_end      = t_end_np
                last_in_alert = safe_last(next_out, i)
                alert_log.append({'airport': airport, 'alert_start': alert_start_ts,
                                        'alert_end': pd.Timestamp(t_end_np), 'type': 'p10'})
                next_check = alert_end
                i+=1
                continue
            else:
                next_check = t_now + _INIT_NS
                i += 1
                continue


        if alert_end is not None:
            alert_log.append({'airport': airport,
                              'alert_start': pd.Timestamp(alert_start_ts) if alert_start_ts else None,
                              'alert_end': pd.Timestamp(alert_end), 'type': 'fin_episode'})

    return pd.Series(alerte_arr.astype(int), index=orig_idx), alert_log



In [10]:
print("=== Simulation du système d'alerte ===\n")
data = data.reset_index(drop=True)
groups  = [(airport, grp) for airport, grp in data.groupby('airport')]

# ⑥ Parallélisation inter-aéroports
results = Parallel(n_jobs=-1, backend='loky')(
    delayed(simulate_alert_system)(grp) for _, grp in groups
)

all_alerts = pd.Series(0, index=data.index, dtype=int)
all_logs   = []
for (airport, grp), (result, logs) in zip(groups, results):
    all_alerts.loc[result.index] = result.values
    all_logs.extend(logs)
    n_alert = result.sum()
    print(f"  {airport:10} : {len(grp):>7,} lignes → {n_alert:>6,} en alerte ({n_alert/len(grp)*100:.1f}%)")

data['alerte'] = all_alerts

df_alerts = pd.DataFrame(all_logs)
df_alerts['duree_min'] = (
    (df_alerts['alert_end'] - df_alerts['alert_start']).dt.total_seconds() / 60
).round(1)

print(f"\n=== Log : {len(df_alerts)} alertes ===")
print(df_alerts.groupby(['airport', 'type']).size().unstack(fill_value=0))
print(f"\nDurées (min) :\n{df_alerts['duree_min'].describe().round(1)}")
print(f"\nTaux d'alerte global : {data['alerte'].mean()*100:.1f}%")
print(data.groupby('airport')['alerte'].mean().mul(100).round(1).sort_values(ascending=False).to_string())


=== Simulation du système d'alerte ===

  Ajaccio    :  21,331 lignes → 17,232 en alerte (80.8%)
  Bastia     :  42,291 lignes → 37,379 en alerte (88.4%)
  Biarritz   :  50,477 lignes → 46,389 en alerte (91.9%)
  Nantes     :   7,556 lignes →  6,083 en alerte (80.5%)
  Pise       :  66,520 lignes → 61,618 en alerte (92.6%)

=== Log : 8299 alertes ===
type      CG20  danger_3km  fin_episode  p10  prolongation
airport                                                   
Ajaccio    140           1          155  325           673
Bastia     220           9          179  444          1182
Biarritz   157           7          152  433          1508
Nantes      81           0           74  136           282
Pise       197          26          193  529          1196

Durées (min) :
count    8299.0
mean        9.2
std         9.0
min         0.0
25%         8.6
50%        10.0
75%        10.0
max       220.8
Name: duree_min, dtype: float64

Taux d'alerte global : 89.7%
airport
Pise        92.6
Bia

In [11]:
df_alerts.head()

,airport,alert_start,alert_end,type,duree_min
0,Ajaccio,2023-01-08 23:18:19,2023-01-08 23:28:19,p10,10.0
1,Ajaccio,2023-01-08 23:35:46,2023-01-08 23:45:46,p10,10.0
2,Ajaccio,2023-01-09 01:07:52,2023-01-09 01:17:52,CG20,10.0
3,Ajaccio,2023-01-09 01:35:18,2023-01-09 01:45:18,p10,10.0
4,Ajaccio,2023-01-09 01:35:18,2023-01-09 01:45:18,fin_episode,10.0


In [24]:
#bol = (~df['icloud'])&(df['dist']<20) & (df['airport_alert_id'].isna())# alerte non lancé quand CG20
bol = (df['icloud'])&(df['dist']<20) & (~df['airport_alert_id'].isna())# alerte lancé quand intracloud <20km
sum(bol)

0

In [21]:
data.loc[30:50]

,lightning_id,lightning_airport_id,date,lon,lat,amplitude,maxis,icloud,dist,azimuth,...,airport_alert_id,is_last_lightning_cloud_ground,probas_10,probas_15,probas_30,probas_15_3km,alerte,alert_active_start,alert_active_end,score
30,72532,72532,2023-01-17 07:23:30+00:00,8.9638,41.7990,52.14,0.082,False,19.209596,127.753987,...,532.0,True,0.573129,0.436047,0.673372,0.997869,1,2023-01-17 07:17:34,2023-01-17 07:27:34,0.426871
31,72533,72533,2023-01-17 07:24:25+00:00,8.9946,41.7937,59.84,0.105,False,21.449099,124.122403,...,NaN,<NA>,0.227362,0.284304,0.379634,0.998177,1,2023-01-17 07:17:34,2023-01-17 07:27:34,0.772638
32,72534,72534,2023-01-17 07:25:06+00:00,8.9251,41.8148,-4.33,3.889,True,15.761109,131.680069,...,NaN,<NA>,0.124555,0.249500,0.199348,0.997638,1,2023-01-17 07:17:34,2023-01-17 07:27:34,0.875445
33,72535,72535,2023-01-17 07:25:31+00:00,9.0028,41.7692,5.40,0.131,True,23.836288,127.682002,...,NaN,<NA>,0.119081,0.249597,0.176751,0.998114,1,2023-01-17 07:17:34,2023-01-17 07:27:34,0.880919
34,72536,72536,2023-01-17 07:26:45+00:00,8.9921,41.8071,30.50,0.055,False,20.315540,121.622706,...,NaN,<NA>,0.785903,0.326663,0.606066,0.998958,1,2023-01-17 07:17:34,2023-01-17 07:27:34,0.214097
35,72537,72537,2023-01-17 07:28:03+00:00,9.0558,41.8021,133.12,0.113,False,24.905842,115.660889,...,NaN,<NA>,0.768246,0.595486,0.841416,0.998685,0,NaT,NaT,0.231754
36,72538,72538,2023-01-17 07:28:46+00:00,9.0145,41.7876,46.71,0.082,False,23.132059,122.729752,...,NaN,<NA>,0.767735,0.495933,0.824867,0.999425,0,NaT,NaT,0.232265
37,72539,72539,2023-01-17 07:29:13+00:00,9.0399,41.8053,24.84,0.105,False,23.610604,116.526357,...,NaN,<NA>,0.515445,0.375145,0.533852,0.999555,0,NaT,NaT,0.484555
38,72540,72540,2023-01-17 07:29:40+00:00,9.0470,41.7911,19.22,0.069,False,24.998354,118.493567,...,NaN,<NA>,0.480736,0.532721,0.601557,0.999263,0,2023-01-17 07:29:40,2023-01-17 07:39:40,0.519264
39,72541,72541,2023-01-17 07:30:30+00:00,9.0498,41.7932,73.34,0.100,False,25.050029,117.840732,...,NaN,<NA>,0.772248,0.421314,0.602168,0.999495,1,2023-01-17 07:29:40,2023-01-17 07:39:40,0.227752


In [12]:
import numpy as np

df_alerts_active = df_alerts[df_alerts['type'] != 'fin_episode'].copy()
df_alerts_active = df_alerts_active.sort_values(['airport', 'alert_start']).reset_index(drop=True)

# ── 1. Fusionner les alertes qui se chevauchent / se prolongent ──
merged_alerts = []
for airport, grp in df_alerts_active.groupby('airport'):
    grp = grp.sort_values('alert_start')
    current_start = None
    current_end   = None

    for _, row in grp.iterrows():
        if current_start is None:
            current_start = row['alert_start']
            current_end   = row['alert_end']
        elif row['alert_start'] <= current_end:
            # Chevauchement ou prolongation → étendre
            current_end = max(current_end, row['alert_end'])
        else:
            merged_alerts.append({'airport': airport,
                                  'alert_start': current_start,
                                  'alert_end': current_end})
            current_start = row['alert_start']
            current_end   = row['alert_end']

    if current_start is not None:
        merged_alerts.append({'airport': airport,
                              'alert_start': current_start,
                              'alert_end': current_end})

df_merged = pd.DataFrame(merged_alerts)
print(f"Alertes brutes : {len(df_alerts_active)} → fusionnées : {len(df_merged)}")

# ── 2. Mapping vectorisé avec searchsorted ───────────────────
data['alert_active_start'] = pd.NaT
data['alert_active_end']   = pd.NaT

for airport, grp_alerts in df_merged.groupby('airport'):
    mask     = data['airport'] == airport
    dates_np = data.loc[mask, 'date'].values.astype('datetime64[ns]')

    starts = grp_alerts['alert_start'].values.astype('datetime64[ns]')
    ends   = grp_alerts['alert_end'].values.astype('datetime64[ns]')

    idx_insert = np.searchsorted(starts, dates_np, side='right') - 1

    valid = idx_insert >= 0
    valid[valid] &= dates_np[valid] <= ends[idx_insert[valid]]

    result_start = np.full(len(dates_np), np.datetime64('NaT'), dtype='datetime64[ns]')
    result_end   = np.full(len(dates_np), np.datetime64('NaT'), dtype='datetime64[ns]')

    result_start[valid] = starts[idx_insert[valid]]
    result_end[valid]   = ends[idx_insert[valid]]

    data.loc[mask, 'alert_active_start'] = result_start
    data.loc[mask, 'alert_active_end']   = result_end

print(f"Lignes en alerte avec dates : {data['alert_active_end'].notna().sum():,}")
print(f"Lignes hors alerte          : {data['alert_active_end'].isna().sum():,}")

Alertes brutes : 7546 → fusionnées : 2476
Lignes en alerte avec dates : 170,922
Lignes hors alerte          : 17,253


In [13]:
df_alerts_active = df_alerts[df_alerts['type'] != 'fin_episode'].copy()

data['alert_active_start'] = pd.NaT
data['alert_active_end']   = pd.NaT

for airport, grp_alerts in df_alerts_active.groupby('airport'):
    mask = data['airport'] == airport
    dates_np = data.loc[mask, 'date'].values.astype('datetime64[ns]')

    starts = grp_alerts['alert_start'].values.astype('datetime64[ns]')
    ends   = grp_alerts['alert_end'].values.astype('datetime64[ns]')

    # Pour chaque ligne : trouver la dernière alerte qui a commencé avant cette date
    # searchsorted donne l'index d'insertion → -1 = dernière alerte commencée
    idx_insert = np.searchsorted(starts, dates_np, side='right') - 1

    valid = idx_insert >= 0
    # Vérifier que la date est bien avant la fin de cette alerte
    valid[valid] &= dates_np[valid] <= ends[idx_insert[valid]]

    result_start = np.full(len(dates_np), np.datetime64('NaT'), dtype='datetime64[ns]')
    result_end   = np.full(len(dates_np), np.datetime64('NaT'), dtype='datetime64[ns]')

    result_start[valid] = starts[idx_insert[valid]]
    result_end[valid]   = ends[idx_insert[valid]]

    data.loc[mask, 'alert_active_start'] = result_start
    data.loc[mask, 'alert_active_end']   = result_end

print(f"Lignes en alerte avec dates : {data['alert_active_end'].notna().sum():,}")
print(f"Lignes hors alerte          : {data['alert_active_end'].isna().sum():,}")

Lignes en alerte avec dates : 170,922
Lignes hors alerte          : 17,253


In [14]:
data['score'] = 1- data['probas_10']

In [15]:
# ── Filtrer : garder uniquement les lignes avec une alerte prédite ──
df_submit = data[data['airport_alert_id'].notna()].copy()

# ── Ne garder que les colonnes utiles et renommer ──
df_submit = df_submit[['airport', 'airport_alert_id', 'date', 'alert_active_end', 'score']].rename(columns={
    'date':               'prediction_date',
    'alert_active_end':   'predicted_date_end_alert',
    'score':             'confidence'
})

df_submit = df_submit.reset_index(drop=True)
print(f"Shape : {df_submit.shape}")
df_submit.head()

Shape : (18010, 5)


,airport,airport_alert_id,prediction_date,predicted_date_end_alert,confidence
0,Ajaccio,531.0,2023-01-09 01:07:52+00:00,2023-01-09 01:17:52,0.332067
1,Ajaccio,532.0,2023-01-17 07:17:34+00:00,2023-01-17 07:27:34,0.164643
2,Ajaccio,532.0,2023-01-17 07:21:06+00:00,2023-01-17 07:27:34,0.665430
3,Ajaccio,532.0,2023-01-17 07:23:30+00:00,2023-01-17 07:27:34,0.426871
4,Ajaccio,533.0,2023-01-17 10:52:24+00:00,2023-01-17 11:02:24,0.715228


### Part 1 Generate fake predictions

We simply generate 20 predictions around the true date of the last lightning with a linear growth of the confidence threshold.

### Part 2 Evaluate predictions

In [16]:
import pandas as pd
from tqdm import tqdm

In [17]:
# our 30 minutes baseline
MAX_GAP_MINUTES = 30


# here replace with your own file
input_file = "../data/segment_alerts_all_airports_train.csv"
df = pd.read_csv(input_file)


#building the Thetas
n_samples = 20
thetas = [i/n_samples for i in range(n_samples)]

# min distance at which we consider a lightning is really dangerous for the airport
min_dist = 3
# total number of dangerous lightning
tot_lightnings = len(df.loc[df.dist<min_dist])

# read prediction file and convert to timestampts
predictions = df_submit
predictions.predicted_date_end_alert = pd.to_datetime(predictions.predicted_date_end_alert)

# group by alerts 
alerts = df.groupby(['airport','airport_alert_id'])

The following cell computes the number of missed dangerous lightning and the gain in time for the different thetas

In [18]:
results = {}
df_submit['airport_alert_id'] = df_submit['airport_alert_id'].astype(int)
for theta in tqdm(thetas):
    pred_over_theta = predictions.loc[predictions['confidence'] >= theta ]
    pred_over_theta_min = pred_over_theta.groupby(['airport','airport_alert_id']).predicted_date_end_alert.min()
    gain, missed_lights = 0, 0
    for (airport, alert_id), end_alert_pred in pred_over_theta_min.items():
        lightings = alerts.get_group((airport, alert_id))
        end_alert_baseline = pd.to_datetime(lightings.date, utc=True).max() + pd.Timedelta(minutes=MAX_GAP_MINUTES)
        gain += (end_alert_baseline - end_alert_pred).total_seconds()
        missed_lights += sum(pd.to_datetime(lightings.loc[lightings.dist<min_dist].date, utc=True) > end_alert_pred )
    results[theta] = (gain, missed_lights)

  0%|          | 0/20 [00:00<?, ?it/s]


KeyError: ('Ajaccio', 531)

Perform some visualisations : normally, the higher the Theta, the lower the gain and the lower the risk. This kind of visualisation can be usefull to show to the Jury the different compromises that your model can reach.

In [ ]:
import matplotlib.pyplot as plt
gains = [results[theta][0]/3600 for theta in thetas]
missed = [results[theta][1] / tot_lightnings for theta in thetas]
plt.plot(missed, gains, marker='*', markersize=8)
for t,g,m in zip(thetas,gains,missed):
    plt.text(m, g, str(t))
plt.xlabel('Number of missed lightnings')
plt.ylabel('Gain of time in hours')
plt.title('Gain of time and missed lightnings for different thresholds')

KeyError: 0.0


**Selection of the best theta**

What is the theta with maximize the gain of time and respect the constraint on the acceptable risk ?

In [ ]:
ACCEPTABLE_RISK = 0.02

(gain_of_time, theta, missing_lightnings) = max([ (gain_of_time, theta, missing_rate)  
     for (theta,(gain_of_time, missing_rate)) 
     in results.items() if missing_rate/tot_lightnings < ACCEPTABLE_RISK ])

In [ ]:
print('The system enables to save',
      int(gain_of_time/3600),
      'hours, with a risk of',ACCEPTABLE_RISK,
      'by using a threshold on the confidence score with a value of',
      theta)

### Part 3 Evaluate given a Theta

It is essentially the same code as before, its just that we evaluate it with one value of Theta. So you don't really need this code yourself. But this is what we will do with the predictions.csv file you might send us.


In [ ]:
# redefining some variables
import pandas as pd
predictions = pd.read_csv('predictions.csv')
predictions.predicted_date_end_alert = pd.to_datetime(predictions.predicted_date_end_alert)
MAX_GAP_MINUTES = 30
input_file = "/home/paul/programmation/iapau/databattle2026/databattle2026/data_train_databattle2026_v1/segment_alerts_all_airports_train.csv"
df = pd.read_csv(input_file)
min_dist = 3 # at which distance a strike is considered dangerous
tot_lightnings = len(df.loc[df.dist<min_dist])
alerts = df.groupby(['airport','airport_alert_id'])

theta = 0.4 # using the theta value we obtained from the previous section

In [ ]:
# Essentially the same code as in part 2 of this notebook

pred_over_theta = predictions.loc[predictions['confidence'] >= theta ]
pred_over_theta_min = pred_over_theta.groupby(['airport','airport_alert_id']).predicted_date_end_alert.min()
gain, missed_lights = 0, 0
for (airport, alert_id), end_alert_pred in pred_over_theta_min.items():
    lightings = alerts.get_group((airport, alert_id))
    end_alert_baseline = pd.to_datetime(lightings.date, utc=True).max() + pd.Timedelta(minutes=MAX_GAP_MINUTES)
    gain += (end_alert_baseline - end_alert_pred).total_seconds()
    missed_lights += sum(pd.to_datetime(lightings.loc[lightings.dist<min_dist].date, utc=True) > end_alert_pred )

print('for a theta of',theta,
      'the gain of time is',int(gain/3600),
      'hours, with a rate of missing lightnings',missed_lights/tot_lightnings)

### Conclusion

Feel free to play and modify this notebook in any way you like or judge relevant. 

The distance of 3km was suggested by Sébastien from Meteorage, but it might be interesting to provide results for other values as well.

One thing we wondered but did not consider in the notebook is to compute a gain, not only from the 30 minutes baseline, but also by taking into account the moment of the prediction. Intuitively, for the same prediction, it is better to send the information as soon as possible. With our definition, the Gain is measured only between the baseline and the value of the prediction, and the date is not taken into account. If you worked on this aspect, that's perfectly legitimate!

